# Multi-Object Tracking in Dense Crowds

**CAIC Summer of Technology 2026 — Computer Vision Track, Final Project**

Full pipeline: YOLOv8 detection → ByteTrack association → gap interpolation → HOTA/IDF1 evaluation.

The implementation lives in the `cv_mot` package rather than in notebook cells. That is deliberate:
the association logic is tested (68 unit tests) and reused by the CLI, the tuning sweep and the
ablation study, and copies of an algorithm pasted into cells drift apart. This notebook is the
narrative and the evidence; `src/cv_mot/` is the implementation.

---

## Contents

1. Setup and data inspection
2. Why this dataset is hard — quantifying occlusion
3. The evaluator, and why it is built first
4. Detection: YOLOv8 and the resolution/recall trade-off
5. Association: SORT's failure mode and ByteTrack's fix
6. Isolating association from detection with an oracle detector
7. Tuning the confidence threshold
8. Ablation study
9. Final submission and format validation

## 1. Setup and data inspection

Install with the detection extras, then confirm the sequence layout.

In [ ]:
# !pip install -e '.[detect,dev]'
import pathlib, sys

# Locate the project root by walking up to the directory holding pyproject.toml.
# A notebook's working directory depends on how it was launched, so relying on a
# relative path like '../Week 4/data' breaks under nbconvert and Colab.
PROJECT = next(
    parent for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (parent / 'pyproject.toml').exists()
)
REPO = PROJECT.parent          # holds the original course material
DATA = REPO / 'Week 4' / 'data'
CACHE = REPO / 'cache' / 'yolov8s_1280'
sys.path.insert(0, str(PROJECT / 'src'))

import numpy as np
from cv_mot.mot_io import discover_sequences, read_mot_file, read_seqinfo

print('project:', PROJECT)
sequences = discover_sequences(DATA)
for s in sequences:
    print(f'{s.name:>5}  {s.length:>5} frames  {s.width}x{s.height}  {s.frame_rate} fps  gt={s.gt_path.exists()}')


### Ground truth format — read the columns before trusting them

The course README states that `gt.txt` has `class` always equal to 1. **That is not true of this
data.** These are MOTChallenge sequences and the class column takes many values:

| class | meaning | role in scoring |
|---|---|---|
| 1 | pedestrian | the target |
| 2 | person on vehicle | distractor — predictions here are *deleted* |
| 7 | static person | distractor |
| 8 | distractor (mannequin, poster) | distractor |
| 12 | reflection | distractor |
| 3,4,5,9,10,11 | vehicles, occluders | ignored entirely |

A detector that fires on a shop-window reflection has not made the kind of mistake the benchmark
punishes, so the official protocol deletes such predictions instead of counting them as false
positives. Getting this wrong inflates FP heavily and makes a good tracker look broken.

In [ ]:
import collections
for s in sequences:
    g = read_mot_file(s.gt_path)
    classes = collections.Counter(g[:, 7].astype(int).tolist())
    scored = ((g[:, 6] != 0) & (g[:, 7].astype(int) == 1)).sum()
    print(f'{s.name:>5}  {len(g):>6} rows  {scored:>6} scored (class 1)  classes={dict(sorted(classes.items()))}')

## 2. Why this dataset is hard — quantifying occlusion

Before choosing a method, measure the problem. The `visibility` column says what fraction of each
person is actually visible.

In [ ]:
for s in sequences:
    g = read_mot_file(s.gt_path)
    ped = g[(g[:, 6] != 0) & (g[:, 7].astype(int) == 1)]
    per_frame = collections.Counter(ped[:, 0].astype(int).tolist())
    print(f'{s.name:>5}  {len(ped):>6} dets  peak {max(per_frame.values()):>3}/frame  '
          f'median {np.median(list(per_frame.values())):>5.0f}/frame  '
          f'<50% visible: {100 * (ped[:, 8] < 0.5).mean():>4.1f}%  '
          f'<10% visible: {100 * (ped[:, 8] < 0.1).mean():>4.1f}%')

On the reference sequence **59% of the people you are scored on are less than half visible**.

This single number drives every design decision that follows. A tracker whose association depends
on confident, cleanly-separated detections has no chance here; the method has to be built around
the assumption that most detections are partial and low-scoring.

## 3. The evaluator, and why it is built first

The leaderboard is `(2·IDF1 + HOTA) / 3`. Without a local implementation of those metrics, tuning
is guesswork. So the evaluator comes before the tracker.

- **IDF1** picks one global ground-truth-ID → predicted-ID mapping for the whole sequence, then
  counts agreeing detections. Splitting one person into two IDs loses roughly half their
  detections to *both* IDFN and IDFP — which is why it carries double weight.
- **HOTA** = `mean over α of sqrt(DetA_α · AssA_α)`, separating detection from association and
  averaging over IoU thresholds 0.05…0.95 so no single cutoff dominates.

Implemented from the source papers in `cv_mot/metrics/`. Correctness is established against cases
whose answers are known by hand:

In [ ]:
from cv_mot.metrics import ALPHAS, evaluate

rng = np.random.default_rng(1)
rows = []
for pid in range(1, 5):
    x, y = rng.uniform(0, 800), rng.uniform(0, 800)
    for f in range(1, 61):
        rows.append([f, pid, x + 2 * f, y + f, 60, 140, 1, 1, 1])
gt = np.array(rows, dtype=float)
as_pred = lambda r: np.c_[r[:, :6], -np.ones((len(r), 4))]
score = lambda p: evaluate({'s': (gt, p, 60)})[1]

print('perfect prediction      ', {k: round(score(as_pred(gt))[k], 4) for k in ('DetA','AssA','IDF1','HOTA')})

# Rename every identity at the midpoint: detection untouched, association halved.
sw = gt.copy(); sw[:, 1] = np.where(sw[:, 0] > 30, sw[:, 1] + 100, sw[:, 1])
print('all IDs switched at 50%%  ', {k: round(score(as_pred(sw))[k], 4) for k in ('DetA','AssA','IDF1','HOTA')})

# A 20px shift on a 60px-wide box gives IoU exactly 0.5, so DetA must equal
# the fraction of alpha thresholds at or below 0.5, i.e. 10/19 = 0.5263.
sh = gt.copy(); sh[:, 2] += 20
print(f'20px shift -> DetA {score(as_pred(sh))["DetA"]:.4f}  (predicted {np.mean(ALPHAS <= 0.5 + 1e-12):.4f})')

## 4. Detection: YOLOv8 and the resolution/recall trade-off

Detections are cached to `.npz` once, then reused by every tuning trial. Inference dominates
runtime, so separating it from association is what makes an exhaustive sweep affordable.

The detector's own `conf` floor is set to **0.05**, far below the tracker's thresholds. This is
essential: ByteTrack's second pass needs low-score boxes to exist. Filtering them at the detector
would silently disable the method.

In [ ]:
# Already cached by: python -m cv_mot detect --data '../Week 4/data' \
#     --out ../cache/yolov8s_1280 --weights yolov8s.pt --imgsz 1280 --conf 0.05
from cv_mot.detect import CachedDetections

ref = read_seqinfo(DATA / 'ref')
dets = list(CachedDetections(CACHE / 'ref.npz').run(ref))
counts = np.array([len(d) for d in dets])
all_scores = np.concatenate([d.scores for d in dets if len(d)])
print(f'{counts.sum()} detections, {counts.mean():.1f}/frame (peak {counts.max()})')
print(f'score < 0.5: {100 * (all_scores < 0.5).mean():.1f}%   '
      f'score < 0.25: {100 * (all_scores < 0.25).mean():.1f}%')
print('\nA large share of boxes are low-confidence -- exactly the population ByteTrack recovers.')

## 5. Association: SORT's failure mode and ByteTrack's fix

SORT applies **one** confidence threshold. In a crowd the dominant event is not a missed person but
a *partially occluded* person detected with a low score. SORT discards that box, the track goes
unmatched, and after a few frames the identity dies — a new one is issued on reappearance.

ByteTrack's insight: a low-score box in roughly the right place is strong evidence **if a track is
already there**. Association therefore runs in two passes:

1. high-score boxes → all active *and* lost tracks;
2. low-score boxes → only the tracks still unmatched, and they may **never create** an identity.

That asymmetry is the whole method: low-score boxes sustain identities without importing the false
positives a globally-lowered threshold would bring. Demonstrated on a synthetic case where one
person's confidence collapses for seven frames:

In [ ]:
from cv_mot.bytetrack import ByteTracker, ByteTrackConfig
from cv_mot.sort import SortConfig, SortTracker

def synthetic(tracker, n_frames=30):
    reported = 0
    for f in range(1, n_frames + 1):
        boxes = np.array([[100. + 5 * f, 200., 50., 120.], [700. - 4 * f, 210., 52., 124.]])
        s = np.array([0.9, 0.25 if 10 <= f <= 16 else 0.9])   # person 2 occluded
        reported += len(tracker.update(boxes, s, img_size=(1920, 1080)))
    return reported

b = synthetic(ByteTracker(ByteTrackConfig()))
s = synthetic(SortTracker(SortConfig(det_thresh=0.5)))
print(f'boxes reported over 30 frames -- ByteTrack {b}, SORT {s}  (SORT loses {b - s} to the dip)')

## 6. Isolating association from detection with an oracle detector

Tracking error has two independent sources: the detector missing people, and the association
assigning them wrongly. To measure the second, feed the tracker detections of *known* quality
derived from the ground truth.

This is a diagnostic, not a shortcut — and it answers a question real detections cannot.

In [ ]:
from cv_mot.detect import GroundTruthDetector
from cv_mot.pipeline import run_sequence

gt_ref = read_mot_file(ref.gt_path)
def oracle_score(**kw):
    d = GroundTruthDetector(recall=1.0, noise_std=0.0, drop_invisible_below=0.0, **kw)
    rows, _ = run_sequence(ref, d, lambda: ByteTracker(ByteTrackConfig(frame_rate=30)))
    return evaluate({'ref': (gt_ref, rows, ref.length)})[1]

a = oracle_score(visibility_scores=False)  # perfect boxes, uniform high confidence
b = oracle_score(visibility_scores=True)   # perfect boxes, realistic confidence
print(f'perfect boxes + uniform confidence : IDF1={a["IDF1"]:.4f}  HOTA={a["HOTA"]:.4f}  IDSW={int(a["IDSW"])}')
print(f'perfect boxes + realistic confidence: IDF1={b["IDF1"]:.4f}  HOTA={b["HOTA"]:.4f}  IDSW={int(b["IDSW"])}')
print(f'\ncost of confidence alone: {a["IDF1"] - b["IDF1"]:.4f} IDF1')

Two conclusions, both load-bearing:

1. With perfect boxes the tracker scores **IDF1 0.983 with zero identity switches** — the
   association logic and the Kalman model are sound, so remaining error is not a tracker bug.
2. Changing *only the confidence values* — boxes still perfect — costs **24.2 IDF1 points**.

Confidence handling, not box quality, is the dominant lever on this data. That justifies spending
the tuning budget on thresholds.

## 7. Tuning the confidence threshold

The project brief hints that the optimum for dense crowds is lower than the default. Here is the
measurement rather than the assumption:

In [ ]:
print(f"{'high':>5}{'new':>6}{'IDF1':>9}{'HOTA':>8}{'DetA':>8}{'AssA':>8}{'Score':>8}")
for hi, new in [(0.6, 0.7), (0.5, 0.6), (0.4, 0.5), (0.3, 0.4), (0.25, 0.3), (0.2, 0.25), (0.16, 0.2)]:
    cfg = ByteTrackConfig(frame_rate=30, track_high_thresh=hi, new_track_thresh=new)
    d = GroundTruthDetector(recall=1.0, visibility_scores=True, drop_invisible_below=0.0)
    rows, _ = run_sequence(ref, d, lambda c=cfg: ByteTracker(c))
    o = evaluate({'ref': (gt_ref, rows, ref.length)})[1]
    print(f'{hi:>5.2f}{new:>6.2f}{o["IDF1"]:>9.4f}{o["HOTA"]:>8.4f}{o["DetA"]:>8.4f}{o["AssA"]:>8.4f}{o["Score"]:>8.4f}')

`DetA` keeps climbing as the threshold falls, but `AssA` turns over: past the optimum the extra
boxes start causing mis-associations faster than they add detections. The leaderboard score peaks
where those two curves cross — a real trade-off located by measurement, not a default left in place.

## 8. Ablation study

Each decision measured as a delta on identical detections, so no claim rests on assertion.
Run from the CLI:

```bash
python -m cv_mot ablate --data '../Week 4/data' --seqs ref --dets ../cache/yolov8s_1280
```

See `docs/RESULTS.md` for the full table.

## 9. Final submission and format validation

```bash
python -m cv_mot track --data '../Week 4/data' --dets ../cache/yolov8s_1280 \
    --config configs/final.json --interpolate --out outputs
python -m cv_mot evaluate --data '../Week 4/data' --preds outputs
python scripts/validate_submission.py --data '../Week 4/data' outputs/*.txt
```

The validator exists because the expensive mistakes in this format are silent: 0-based frames,
`xyxy` left unconverted, a `conf` column that is not `-1`. All of those parse cleanly and simply
score near zero.

In [ ]:
import json

from cv_mot.interpolate import interpolate_tracks
from cv_mot.mot_io import write_mot_file

# Load the tuned configuration rather than restating parameters here, so the
# notebook cannot drift from configs/final.json and docs/RESULTS.md.
final_cfg = json.loads((PROJECT / 'configs' / 'final.json').read_text())
cfg = ByteTrackConfig(frame_rate=ref.frame_rate, **final_cfg)

rows, stats = run_sequence(ref, CachedDetections(CACHE / 'ref.npz'),
                           lambda: ByteTracker(cfg),
                           interpolate=True, interpolate_max_gap=20)
print(f'{stats.frames} frames, {stats.output_rows} rows (+{stats.interpolated_rows} interpolated)')
o = evaluate({'ref': (gt_ref, rows, ref.length)})[1]
print(f'IDF1={o["IDF1"]:.4f}  HOTA={o["HOTA"]:.4f}  Score={o["Score"]:.4f}')
print('matches docs/RESULTS.md ref row (IDF1 0.4182, Score 0.3937):',
      abs(o['IDF1'] - 0.4182) < 1e-4 and abs(o['Score'] - 0.3937) < 1e-4)
write_mot_file(PROJECT / 'outputs' / 'ref.txt', rows[:, :6])
